# boolean-mask-combine composite — cx30: stack per-predicate masks, then AND-reduce across the new axis

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `boolean-mask-combine`, `stack-vs-cat`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "boolean-mask-combine"
DD_ATOM_IDS = ["boolean-mask-combine", "stack-vs-cat"]
DD_SUBTOPICS = ["Numpy: Boolean mask combine", "PyTorch: stack vs cat"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

When you have **K predicates** that all live on the same shape `S`, two patterns get the joint AND:
- **Pairwise chain** — `m0 & m1 & m2 & ...`. Fine for K=2, gets unwieldy at K=5.
- **Stack-and-reduce** — `t.stack([m0, m1, m2, ...], dim=0).all(dim=0)`. Scales to any K, and the stacked tensor is itself a useful debug artifact (`(K, *S)` lets you see WHICH predicate vetoed which slice).

The key choice is `stack` vs `cat`:
- `t.stack(tensors, dim=0)` — INSERTS a new axis of length `K`. Output shape `(K, *S)`. Use this when each tensor is one "slot".
- `t.cat(tensors, dim=0)` — CONCATENATES along an existing axis. Output shape `(K*S[0], *S[1:])`. Use this when you're growing an existing axis.

For the K-mask AND, you want `stack` — you want a NEW axis to reduce over, not a longer version of the first axis.

**Anatomy.**
- `stacked = t.stack(masks, dim=0)`  → `(K, *S)` boolean tensor.
- `combined = stacked.all(dim=0)`    → `(*S,)` boolean tensor — atom: boolean-mask-combine.

Reducing with `.all` is mathematically equivalent to chaining `&`, but the stack form is K-agnostic and lets you inspect the per-predicate breakdown for free.

### Composite Exercise — stack per-predicate masks, then AND-reduce across the new axis

**Atoms exercised together**: `boolean-mask-combine`, `stack-vs-cat`

Implement two functions.

1. `cx30_stack_masks(masks)` — take a Python list of K boolean tensors, each shape `S`. Return a `(K, *S)` boolean tensor via `t.stack`. (Why `stack` and not `cat`: stack inserts a NEW axis of length K; cat would concatenate along an existing axis and lose the per-predicate identity.)
2. `cx30_and_reduce(masks)` — same input. Return a single shape-`S` boolean tensor that is True iff every mask is True at that position. Implementation: stack with `cx30_stack_masks`, then `.all(dim=0)`.

The two-step decomposition exercises both atoms cleanly: `stack` constructs the joint tensor, `boolean-mask-combine` (via `.all`) collapses it.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx30_stack_masks(masks):
    raise NotImplementedError

def cx30_and_reduce(masks):
    raise NotImplementedError

def _test_cx30():
    # Case A: K=3, S=(4,). Stack must produce (3, 4); AND-reduce must produce (4,).
    m0 = t.tensor([True, True, True, False])
    m1 = t.tensor([True, True, False, True])
    m2 = t.tensor([True, False, True, True])
    stacked = cx30_stack_masks([m0, m1, m2])
    assert stacked.dtype == t.bool
    assert tuple(stacked.shape) == (3, 4), f'expected (3,4), got {tuple(stacked.shape)}'
    # Per-predicate identity must be preserved.
    assert t.equal(stacked[0], m0)
    assert t.equal(stacked[1], m1)
    assert t.equal(stacked[2], m2)
    combined = cx30_and_reduce([m0, m1, m2])
    assert combined.dtype == t.bool
    assert tuple(combined.shape) == (4,)
    assert combined.tolist() == [True, False, False, False]

    # Case B: 2-D shape, K=2. Verify stack yields (K, H, W).
    rng = t.Generator().manual_seed(30)
    H, W = 5, 6
    m0 = t.rand(H, W, generator=rng) > 0.3
    m1 = t.rand(H, W, generator=rng) > 0.3
    stacked = cx30_stack_masks([m0, m1])
    assert tuple(stacked.shape) == (2, H, W)
    combined = cx30_and_reduce([m0, m1])
    assert tuple(combined.shape) == (H, W)
    assert t.equal(combined, m0 & m1)

    # Case C: K=5 ARENA-shaped predicates over a (NR, NT)=(8, 7) grid — the canonical use case.
    rng = t.Generator().manual_seed(31)
    NR, NT = 8, 7
    masks = [t.rand(NR, NT, generator=rng) > 0.2 for _ in range(5)]
    stacked = cx30_stack_masks(masks)
    assert tuple(stacked.shape) == (5, NR, NT)
    combined = cx30_and_reduce(masks)
    assert tuple(combined.shape) == (NR, NT)
    # Cross-check against the pairwise chain.
    ref = masks[0]
    for m in masks[1:]:
        ref = ref & m
    assert t.equal(combined, ref)

    # Case D: stack-vs-cat sanity — the result of stack on K shape-(4,) tensors has shape (K, 4),
    # NOT (K*4,). If a candidate accidentally used cat we'd see (12,) here.
    m0 = t.tensor([True, False, True, True])
    m1 = t.tensor([False, True, True, True])
    m2 = t.tensor([True, True, False, True])
    stacked = cx30_stack_masks([m0, m1, m2])
    assert tuple(stacked.shape) == (3, 4), (
        f'expected (3, 4) from stack — did you accidentally use cat? Got {tuple(stacked.shape)}'
    )
    _dd_passed.add('cx30')

_test_cx30()

<details><summary>Show solution — cx30</summary>

```python
def cx30_stack_masks(masks):
    # Atom A (stack-vs-cat): t.stack INSERTS a new dim 0 of length K = len(masks).
    # cat would have concatenated along an existing dim and lost the per-predicate axis.
    return t.stack(masks, dim=0)

def cx30_and_reduce(masks):
    stacked = cx30_stack_masks(masks)
    # Atom B (boolean-mask-combine): collapse the new K-axis with logical-AND.
    # Equivalent to chaining `m0 & m1 & ... & m_{K-1}` but K-agnostic.
    return stacked.all(dim=0)
```

Why stack instead of cat: each mask is its own predicate — we want them on SEPARATE slots of a new axis so the per-predicate breakdown is recoverable (`stacked[i]` is predicate i). Cat would have concatenated along an existing axis and the per-predicate identity would be lost. The `.all(dim=0)` is the same operation as `m0 & m1 & ... & m_{K-1}` but K-agnostic and amenable to debugging — print `stacked.all(dim=(1, 2))` to see how many entries each predicate kept.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx30'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx30',
        'subtopics': ["Numpy: Boolean mask combine", "PyTorch: stack vs cat"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()